# Notebook 03 — Avaliação e Benchmarks de Compressão

## Aula 04: Otimização de Modelos — Pruning e Quantização

---

### 🎯 Objetivos deste Notebook

1. Carregar todos os modelos produzidos no Notebook 02
2. Comparar **acurácia**, **latência** e **tamanho** em gráficos unificados
3. Avaliar **acurácia por classe** para detectar degradação seletiva
4. Analisar os **trade-offs** discutidos na seção *Saiba Mais* do doc 04
5. Reproduzir os benchmarks da Tabela 2 (Jacob et al., 2018) em escala pedagógica
6. Gerar relatório consolidado de métricas

### 📖 Conceito Teórico (Documento 04 — Trade-offs)

> A compressão de modelos invariavelmente envolve trade-offs entre métricas de
> eficiência e qualidade de predição. Empiricamente, observa-se que a relação
> entre esparsidade e degradação de acurácia é frequentemente **não-linear**,
> com compressões moderadas (até 50-70%) causando impacto mínimo, seguidas de
> degradação acelerada para compressões mais agressivas.
>
> — Blalock et al. (2020), *What is the State of Neural Network Pruning?*

---

In [ ]:
# Imports necessários
import sys
import copy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn

# Adicionar diretório raiz ao path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_preprocessing import DataPreprocessor, CIFAR10_CLASSES
from src.model import (
    SimpleCNN,
    StudentCNN,
    apply_unstructured_pruning,
    remove_pruning_reparametrization,
    compute_sparsity,
    quantize_dynamic,
    count_parameters,
    get_model_size_mb,
)
from src.training import train_model, fine_tune, evaluate
from src.evaluation import (
    calculate_accuracy,
    calculate_per_class_accuracy,
    calculate_metrics,
    measure_latency,
    plot_compression_comparison,
    plot_sparsity_distribution,
    plot_weight_distribution,
    plot_training_history,
)
from src.utils import save_model, load_model, save_metrics, set_seed, get_device

# Configurações
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
set_seed(42)
device = get_device()
print(f"Dispositivo: {device}")

## 1. Carregar Dados e Modelos

Carregamos o CIFAR-10 e reconstruimos os modelos. Se os checkpoints do
Notebook 02 estiverem disponíveis, carregamos os pesos; caso contrário,
treinamos rapidamente modelos mínimos para demonstração.

In [ ]:
# Dados
preprocessor = DataPreprocessor(data_dir="data/raw", batch_size=128, seed=42)
preprocessor.load_data(download=True)
train_loader, val_loader, test_loader = preprocessor.split_data(val_fraction=0.1)

# Modelo base
model_base = SimpleCNN(num_classes=10)
baseline_path = Path("outputs/models/baseline.pt")
if baseline_path.exists():
    model_base = load_model(model_base, str(baseline_path), device=str(device))
    print("Modelo baseline carregado de checkpoint.")
else:
    print("Checkpoint não encontrado — treinando modelo base (3 épocas)...")
    _ = train_model(model_base, train_loader, val_loader, epochs=3, lr=0.01, device=device)

model_base = model_base.to(device)
model_base.eval()
print(f"Baseline params: {count_parameters(model_base):,}")

## 2. Preparar Variantes de Modelo

Criamos todas as variantes para benchmark comparativo:
- **Baseline** (FP32 denso)
- **Pruned 30%** (não-estruturado, sem fine-tune)
- **Pruned 50% + Fine-tune**
- **Quantizado INT8** (dinâmico)
- **Pipeline** (Pruning 50% + FT + Quantização)
- **Student** (Knowledge Distillation)

In [ ]:
# Variante 1: Pruned 30% (sem fine-tune)
model_p30 = copy.deepcopy(model_base)
model_p30 = apply_unstructured_pruning(model_p30, amount=0.3)
model_p30 = remove_pruning_reparametrization(model_p30)

# Variante 2: Pruned 50% + fine-tune
model_p50ft = copy.deepcopy(model_base)
model_p50ft = apply_unstructured_pruning(model_p50ft, amount=0.5)
_ = fine_tune(model_p50ft, train_loader, val_loader, epochs=2, lr=0.001, device=device)
model_p50ft = remove_pruning_reparametrization(model_p50ft)

# Variante 3: Quantização dinâmica INT8
model_q8 = quantize_dynamic(copy.deepcopy(model_base).cpu())

# Variante 4: Pipeline (pruning + FT + quantize)
model_pipe = copy.deepcopy(model_base)
model_pipe = apply_unstructured_pruning(model_pipe, amount=0.5)
_ = fine_tune(model_pipe, train_loader, val_loader, epochs=2, lr=0.001, device=device)
model_pipe = remove_pruning_reparametrization(model_pipe)
model_pipe = quantize_dynamic(model_pipe.cpu())

# Variante 5: Student (treinar ou carregar)
model_student = StudentCNN(num_classes=10)
student_path = Path("outputs/models/student_distilled.pt")
if student_path.exists():
    model_student = load_model(model_student, str(student_path), device=str(device))
    print("Student carregado de checkpoint.")
else:
    print("Treinando student direto (sem distillation) para benchmark...")
    _ = train_model(model_student, train_loader, val_loader, epochs=3, lr=0.01, device=device)

print("Todas variantes prontas.")

## 3. Benchmark Completo: Acurácia, Latência e Tamanho

Reproduzimos a análise tríplice de trade-offs discutida no doc 04.
Conforme a Tabela 2 (Jacob et al., 2018), espera-se:
- INT8: 4× redução de memória, 2-4× speedup, < 1% degradação

In [ ]:
# Coletar métricas para todas variantes
# Nota: latência medida em CPU para comparação justa com modelos quantizados
eval_device = "cpu"

models = {
    "Baseline (FP32)": model_base.cpu(),
    "Pruned 30%": model_p30.cpu(),
    "Pruned 50% + FT": model_p50ft.cpu(),
    "Quantized INT8": model_q8,
    "Pipeline (P+FT+Q)": model_pipe,
    "Student (Distill)": model_student.cpu(),
}

results = {}
for name, model in models.items():
    model.eval()
    metrics = calculate_metrics(model, test_loader, device=eval_device)
    results[name] = metrics
    print(f"{name:<22} | Acc: {metrics['accuracy']:.4f} | "
          f"Lat: {metrics['latency_mean_ms']:.2f} ms | "
          f"Size: {metrics['size_mb']:.2f} MB")

In [ ]:
# Tabela consolidada
df_results = pd.DataFrame(results).T
df_results.index.name = "Modelo"
df_results = df_results.round(4)
print(df_results.to_string())

In [ ]:
# Gráfico comparativo — reproduz análise de trade-offs do doc 04
plot_compression_comparison(
    results,
    save_path="outputs/figures/compression_comparison.png",
)

## 4. Análise de Esparsidade — Pruning por Camada

Conforme Blalock et al. (2020) e a Hipótese da Lottery Ticket:
> A sensibilidade varia substancialmente entre camadas, com camadas
> iniciais e finais tipicamente mais sensíveis que camadas intermediárias.

In [ ]:
# Esparsidade por camada no modelo Pruned 50% + FT
# Precisamos recalcular pois remove_pruning zera os pesos permanentemente
model_sp_analysis = copy.deepcopy(model_base)
model_sp_analysis = apply_unstructured_pruning(model_sp_analysis, amount=0.5)
sparsity_50 = compute_sparsity(model_sp_analysis)

plot_sparsity_distribution(
    sparsity_50,
    save_path="outputs/figures/sparsity_per_layer_50.png",
)

## 5. Acurácia por Classe — Detecção de Degradação Seletiva

Verificamos se a compressão afeta classes específicas de forma
desproporcional — importante para fairness em produção.

In [ ]:
# Acurácia per-class para baseline e pipeline comprimido
per_class_base = calculate_per_class_accuracy(models["Baseline (FP32)"], test_loader, device=eval_device)
per_class_pipe = calculate_per_class_accuracy(models["Pipeline (P+FT+Q)"], test_loader, device=eval_device)

df_per_class = pd.DataFrame({
    "Classe": CIFAR10_CLASSES,
    "Baseline": [per_class_base[i] for i in range(10)],
    "Pipeline": [per_class_pipe[i] for i in range(10)],
})
df_per_class["Δ Acc"] = df_per_class["Pipeline"] - df_per_class["Baseline"]
print(df_per_class.to_string(index=False))

In [ ]:
# Gráfico de acurácia por classe
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(10)
width = 0.35
bars1 = ax.bar(x - width/2, df_per_class["Baseline"], width, label="Baseline", color="steelblue")
bars2 = ax.bar(x + width/2, df_per_class["Pipeline"], width, label="Pipeline", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(CIFAR10_CLASSES, rotation=30)
ax.set_ylabel("Acurácia")
ax.set_title("Acurácia por Classe: Baseline vs. Pipeline Comprimido")
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("outputs/figures/per_class_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Curva Esparsidade vs. Acurácia

Reproduzimos a análise de Blalock et al. (2020): a relação é **não-linear**,
com degradação acelerada após ~70% de esparsidade para pruning não-estruturado.

In [ ]:
# Sweeping de esparsidade
amounts = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
sweep_results = []

for amt in amounts:
    m = copy.deepcopy(model_base).cpu()
    if amt > 0:
        m = apply_unstructured_pruning(m, amount=amt)
    acc = calculate_accuracy(m, test_loader, device=eval_device)
    sp = compute_sparsity(m)["global"] if amt > 0 else 0.0
    sweep_results.append({"amount": amt, "sparsity": sp, "accuracy": acc})

df_sweep = pd.DataFrame(sweep_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_sweep["sparsity"], df_sweep["accuracy"], "o-", color="steelblue", linewidth=2)
ax.set_xlabel("Esparsidade (%)")
ax.set_ylabel("Acurácia")
ax.set_title("Esparsidade vs. Acurácia — Pruning Não-Estruturado\n(Blalock et al., 2020)")
ax.grid(True, alpha=0.3)
ax.axhline(df_sweep["accuracy"].iloc[0], color="red", linestyle="--", alpha=0.5, label="Baseline")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/figures/sparsity_vs_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Distribuição de Pesos: Antes vs. Depois da Compressão

Visualizamos o efeito do pruning na distribuição de pesos (pico em zero)
e da quantização (discretização dos valores).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Baseline
w_base = torch.cat([p.data.flatten() for p in models["Baseline (FP32)"].parameters()]).numpy()
axes[0].hist(w_base, bins=150, alpha=0.7, color="steelblue", edgecolor="none")
axes[0].set_title("Baseline (FP32)")
axes[0].set_xlabel("Peso")

# Pruned 50% + FT
w_pruned = torch.cat([p.data.flatten() for p in models["Pruned 50% + FT"].parameters()]).numpy()
axes[1].hist(w_pruned, bins=150, alpha=0.7, color="coral", edgecolor="none")
axes[1].set_title("Pruned 50% + Fine-tune")
axes[1].set_xlabel("Peso")
pct_zero = 100 * np.sum(np.abs(w_pruned) < 1e-6) / w_pruned.size
axes[1].text(0.95, 0.95, f"zeros: {pct_zero:.1f}%",
             transform=axes[1].transAxes, ha="right", va="top")

# Student
w_student = torch.cat([p.data.flatten() for p in models["Student (Distill)"].parameters()]).numpy()
axes[2].hist(w_student, bins=150, alpha=0.7, color="seagreen", edgecolor="none")
axes[2].set_title("Student (Distillation)")
axes[2].set_xlabel("Peso")

plt.suptitle("Distribuição de Pesos — Comparação de Técnicas de Compressão", fontsize=13)
plt.tight_layout()
plt.savefig("outputs/figures/weight_dist_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Benchmark de Latência Detalhado

Medimos percentis de latência (p50, p99) para cada variante,
conforme as boas práticas de benchmarking discutidas nas aulas anteriores.

In [ ]:
latency_details = {}
for name, model in models.items():
    model.eval()
    lat = measure_latency(model, device=eval_device, n_warmup=20, n_runs=200)
    latency_details[name] = lat

df_lat = pd.DataFrame(latency_details).T
df_lat = df_lat.round(3)
print(df_lat.to_string())

In [ ]:
# Gráfico de latência com barras de erro (std)
fig, ax = plt.subplots(figsize=(12, 5))
names = list(latency_details.keys())
means = [latency_details[n]["mean_ms"] for n in names]
stds = [latency_details[n]["std_ms"] for n in names]
p99s = [latency_details[n]["p99_ms"] for n in names]

x = np.arange(len(names))
bars = ax.bar(x, means, yerr=stds, capsize=4, color=sns.color_palette("husl", len(names)))
ax.scatter(x, p99s, marker="D", color="red", zorder=5, label="P99")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=25, ha="right")
ax.set_ylabel("Latência (ms)")
ax.set_title("Latência de Inferência — Comparação de Modelos Comprimidos")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("outputs/figures/latency_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Relatório Final de Métricas

Salvamos todas as métricas em JSON para reprodutibilidade.

In [ ]:
# Salvar relatório consolidado
report = {name: {**results[name], **latency_details[name]} for name in models}
save_metrics(report, "outputs/logs/evaluation_report.json")
print("Relatório salvo em outputs/logs/evaluation_report.json")

# Resumo final
print(f"\n{'='*65}")
print(f"  RESUMO FINAL — Compressão de Modelos (Aula 04)")
print(f"{'='*65}")
for name in models:
    r = results[name]
    print(f"  {name:<22} | Acc: {r['accuracy']:.4f} | "
          f"Size: {r['size_mb']:.2f} MB | "
          f"Lat: {r['latency_mean_ms']:.2f} ms")
print(f"{'='*65}")

## 10. Conclusões e Reflexões

### Resultados Obtidos

Os benchmarks confirmam os trade-offs discutidos na seção *Saiba Mais* do doc 04:

1. **Pruning não-estruturado** até ~50-70% de esparsidade mantém acurácia estável
   (Blalock et al., 2020)
2. **Quantização INT8** reduz tamanho em ~4× com degradação mínima, conforme a
   Tabela 2 do doc 04 (Jacob et al., 2018)
3. **Knowledge Distillation** permite que um modelo ~4× menor capture *dark
   knowledge* do teacher (Hinton et al., 2015)
4. O **pipeline combinado** (Pruning + Fine-tune + Quantização) maximiza a
   compressão, conforme Han et al. (2015)

### Limitações

- Treinamento pedagógico com poucas épocas (resultados abaixo do SOTA)
- Latência medida em CPU (resultados em GPU podem diferir)
- Quantização estática (com calibração) não demonstrada neste notebook

### Referências Principais

- Han, S. et al. (2015). *Deep Compression*. ICLR.
- Frankle, J. & Carlin, M. (2019). *The Lottery Ticket Hypothesis*. ICLR.
- Hinton, G. et al. (2015). *Distilling the Knowledge in a Neural Network*.
- Jacob, B. et al. (2018). *Quantization and Training of Neural Networks*. CVPR.
- Blalock, D. et al. (2020). *What is the State of Neural Network Pruning?* MLSys.